In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WeatherAnalysis") \
    .getOrCreate()

spark

In [5]:
import os
from pyspark.sql.functions import *

base_path = "C:/Users/natha/weather"
all_dfs = []

for year in range(2015, 2025):
    year_path = os.path.join(base_path, str(year))
    for station in ["72429793812", "99495199999"]:
        file_path = os.path.join(year_path, f"{station}.csv")
        if os.path.exists(file_path):
            df = spark.read.csv(file_path, header=True, inferSchema=True)
            df = df.withColumn("YEAR", lit(year))
            all_dfs.append(df)

weather_df = all_dfs[0]
for df in all_dfs[1:]:
    weather_df = weather_df.union(df)

In [6]:
for year in range(2015, 2025):
    for station in ["72429793812", "99495199999"]:
        path = f"C:/weather/{year}/{station}.csv"
        if os.path.exists(path):
            df = spark.read.csv(path, header=True, inferSchema=True)
            print(f"{year} - {station} - Count:", df.count())

In [30]:
from pyspark.sql.window import Window

window = Window.partitionBy("YEAR").orderBy(col("MAX").desc())

hottest = weather_df.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") == 1) \
 .select("STATION", "NAME", "DATE", "MAX", "YEAR")

hottest.show()

+-----------+--------------------+----------+------+----+
|    STATION|                NAME|      DATE|   MAX|YEAR|
+-----------+--------------------+----------+------+----+
|72429793812|CINCINNATI MUNICI...|2015-06-12|  91.9|2015|
|72429793812|CINCINNATI MUNICI...|2016-07-24|  93.9|2016|
|99495199999|SEBASTIAN INLET S...|2017-02-22|9999.9|2017|
|72429793812|CINCINNATI MUNICI...|2018-07-04|  96.1|2018|
|72429793812|CINCINNATI MUNICI...|2019-09-30|  95.0|2019|
|72429793812|CINCINNATI MUNICI...|2020-07-05|  93.9|2020|
|72429793812|CINCINNATI MUNICI...|2021-08-12|  95.0|2021|
|72429793812|CINCINNATI MUNICI...|2022-12-23|9999.9|2022|
|72429793812|CINCINNATI MUNICI...|2023-08-23|  96.1|2023|
|72429793812|CINCINNATI MUNICI...|2024-08-30| 100.9|2024|
+-----------+--------------------+----------+------+----+



In [8]:
march_df = weather_df.filter(month("DATE") == 3)

coldest = march_df.orderBy("MIN").select("STATION", "NAME", "DATE", "MIN").limit(1)

coldest.show()

+-----------+--------------------+----------+---+
|    STATION|                NAME|      DATE|MIN|
+-----------+--------------------+----------+---+
|72429793812|CINCINNATI MUNICI...|2015-03-06|3.2|
+-----------+--------------------+----------+---+



In [33]:
clean_df = weather_df.withColumn(
    "PRCP",
    when(col("PRCP") == 99.99, None).otherwise(col("PRCP"))
)

precip = clean_df.groupBy("STATION", "NAME", "YEAR") \
    .agg(avg("PRCP").alias("Mean_PRCP"))

from pyspark.sql.window import Window

window = Window.partitionBy("STATION").orderBy(col("Mean_PRCP").desc())

most_precip = precip.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") == 1) \
 .select("STATION", "NAME", "YEAR", "Mean_PRCP")

most_precip.show()

+-----------+--------------------+----+-------------------+
|    STATION|                NAME|YEAR|          Mean_PRCP|
+-----------+--------------------+----+-------------------+
|72429793812|CINCINNATI MUNICI...|2018|0.15789041095890405|
|99495199999|SEBASTIAN INLET S...|2015|                0.0|
+-----------+--------------------+----+-------------------+



In [10]:
gust_2024 = weather_df.filter(col("YEAR") == 2024)

result = gust_2024.groupBy("STATION") \
    .agg((sum(col("GUST").isNull().cast("int")) / count("*") * 100).alias("Missing_GUST_Percentage"))

result.show()

+-----------+-----------------------+
|    STATION|Missing_GUST_Percentage|
+-----------+-----------------------+
|72429793812|                    0.0|
|99495199999|                    0.0|
+-----------+-----------------------+



In [11]:
cincy_2020 = weather_df.filter(
    (col("STATION") == "72429793812") & 
    (col("YEAR") == 2020)
)

monthly_stats = cincy_2020.groupBy(month("DATE").alias("MONTH")) \
    .agg(
        avg("TEMP").alias("Mean"),
        expr("percentile_approx(TEMP, 0.5)").alias("Median"),
        stddev("TEMP").alias("StdDev")
    )

monthly_stats.show()

+-----+------------------+------+-----------------+
|MONTH|              Mean|Median|           StdDev|
+-----+------------------+------+-----------------+
|   12| 35.99354838709677|  35.2|6.642787340861814|
|    1| 37.94516129032259|  37.7|8.345810873712928|
|    6| 72.54666666666667|  73.7|4.899946047087439|
|    3|  49.0741935483871|  47.8|8.779406500135623|
|    5| 60.89032258064518|  63.7|9.314768017820217|
|    9|              66.1|  65.8|7.118262089331474|
|    4|51.779999999999994|  51.0|7.313162436838541|
|    8| 73.34516129032258|  73.7|3.487868375734898|
|    7|              77.6|  77.9| 2.33794781806609|
|   10|55.193548387096776|  54.0| 6.72869157582517|
|   11|48.003333333333345|  47.7|6.825938527529321|
|    2|  36.5896551724138|  36.0| 7.90159770587055|
+-----+------------------+------+-----------------+



In [12]:
cincy_2017 = weather_df.filter(
    (col("STATION") == "72429793812") &
    (col("YEAR") == 2017) &
    (col("TEMP") < 50) &
    (col("WDSP") > 3)
)

wc_df = cincy_2017.withColumn(
    "Wind_Chill",
    35.74 + 
    0.6215 * col("TEMP") -
    35.75 * pow(col("WDSP"), 0.16) +
    0.4275 * col("TEMP") * pow(col("WDSP"), 0.16)
)

wc_df.orderBy("Wind_Chill").select("DATE", "Wind_Chill").show(10)

+----------+-------------------+
|      DATE|         Wind_Chill|
+----------+-------------------+
|2017-01-07|-0.4140156367932173|
|2017-12-31| 2.0339767075993116|
|2017-12-27|  3.820645509123832|
|2017-12-28|  4.533355269061226|
|2017-01-06|  4.868933041653884|
|2017-01-08|  7.929748208036862|
|2017-12-25| 14.285113218297408|
|2017-12-30| 14.539211253038193|
|2017-01-05| 14.748861828163854|
|2017-12-26| 15.688977805634499|
+----------+-------------------+
only showing top 10 rows


In [35]:
florida = weather_df.filter(col("STATION") == 99495199999)

extreme_days = florida.filter(col("FRSHTT") != 0)

extreme_days.count()

0

In [24]:
from pyspark.sql.functions import col, when, isnan, month, dayofyear, avg

train = weather_df.filter(
    (col("STATION") == 72429793812) &
    (col("YEAR").isin([2022, 2023])) &
    (month("DATE").isin([11, 12]))
)

train = train.withColumn(
    "MAX",
    when((col("MAX") == 9999.9) | isnan(col("MAX")), None).otherwise(col("MAX"))
)

train = train.dropna(subset=["MAX"])

train = train.withColumn("DAY", dayofyear("DATE").cast("double"))

train.select("DAY", "MAX").show(5)

+-----+----+
|  DAY| MAX|
+-----+----+
|305.0|66.9|
|306.0|66.0|
|307.0|73.0|
|308.0|75.9|
|309.0|75.9|
+-----+----+
only showing top 5 rows


In [25]:
from pyspark.sql.functions import mean

stats = train.select(
    mean("DAY").alias("mean_x"),
    mean("MAX").alias("mean_y")
).collect()[0]

mean_x = stats["mean_x"]
mean_y = stats["mean_y"]

train = train.withColumn(
    "xy",
    (col("DAY") - mean_x) * (col("MAX") - mean_y)
)

train = train.withColumn(
    "xx",
    (col("DAY") - mean_x) * (col("DAY") - mean_x)
)

cov = train.select(mean("xy")).collect()[0][0]
var = train.select(mean("xx")).collect()[0][0]

beta1 = cov / var
beta0 = mean_y - beta1 * mean_x

print("Slope:", beta1)
print("Intercept:", beta0)

Slope: -0.3451964193526393
Intercept: 171.0292771672188


In [26]:
nov_prediction = beta0 + beta1 * 320
dec_prediction = beta0 + beta1 * 349

print("Predicted November 2024 MAX:", nov_prediction)
print("Predicted December 2024 MAX:", dec_prediction)

Predicted November 2024 MAX: 60.56642297437422
Predicted December 2024 MAX: 50.555726813147686


In [28]:
output_path = "C:/Users/natha/weather/natha_results.txt"

with open(output_path, "w") as f:
    f.write("NCEI Weather Analysis Results\n")
    f.write("=============================\n\n")

    # Example: Write Prediction Results
    f.write("Prediction Results:\n")
    f.write(f"Predicted November 2024 MAX: {nov_prediction}\n")
    f.write(f"Predicted December 2024 MAX: {dec_prediction}\n\n")

print("Results file created.")

Results file created.


In [37]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import os

output_path = "C:/Users/natha/weather/nguye3np_results.txt"

with open(output_path, "w") as f:

    f.write("NCEI Global Surface Summary of the Day Analysis\n")
    f.write("==============================================\n\n")

    # -------------------------------------------------
    # 1. Dataset Counts
    # -------------------------------------------------
    f.write("1. Dataset Counts\n")
    f.write("------------------\n")

    total_files = 0
    for year in range(2015, 2025):
        for station in [72429793812, 99495199999]:
            path = f"C:/Users/natha/weather/{year}/{station}.csv"
            if os.path.exists(path):
                df = spark.read.csv(path, header=True, inferSchema=True)
                count = df.count()
                f.write(f"{year} - {station} - Count: {count}\n")
                total_files += 1

    f.write(f"\nTotal files loaded: {total_files}\n\n")


    # -------------------------------------------------
    # 2. Hottest Day Each Year (10 Results)
    # -------------------------------------------------
    f.write("2. Hottest Day Each Year\n")
    f.write("-------------------------\n")

    window_year = Window.partitionBy("YEAR").orderBy(col("MAX").desc())

    hottest = weather_df.withColumn(
        "rank",
        row_number().over(window_year)
    ).filter(col("rank") == 1) \
     .select("STATION", "NAME", "DATE", "MAX", "YEAR")

    for row in hottest.collect():
        f.write(str(row) + "\n")

    f.write("\n")


    # -------------------------------------------------
    # 3. Coldest March Day (2015-2024)
    # -------------------------------------------------
    f.write("3. Coldest March Day (2015-2024)\n")
    f.write("---------------------------------\n")

    march = weather_df.filter(month("DATE") == 3)
    coldest = march.orderBy(col("MIN")).limit(1) \
        .select("STATION", "NAME", "DATE", "MIN")

    for row in coldest.collect():
        f.write(str(row) + "\n")

    f.write("\n")


    # -------------------------------------------------
    # 4. Year with Most Precipitation (Per Station)
    # -------------------------------------------------
    f.write("4. Year with Most Precipitation\n")
    f.write("--------------------------------\n")

    clean_df = weather_df.withColumn(
        "PRCP",
        when(col("PRCP") == 99.99, None).otherwise(col("PRCP"))
    )

    precip = clean_df.groupBy("STATION", "NAME", "YEAR") \
        .agg(avg("PRCP").alias("Mean_PRCP"))

    window_precip = Window.partitionBy("STATION").orderBy(col("Mean_PRCP").desc())

    most_precip = precip.withColumn(
        "rank",
        row_number().over(window_precip)
    ).filter(col("rank") == 1) \
     .select("STATION", "NAME", "YEAR", "Mean_PRCP")

    for row in most_precip.collect():
        f.write(str(row) + "\n")

    f.write("\n")


    # -------------------------------------------------
    # 5. Missing GUST Percentage (2024)
    # -------------------------------------------------
    import pyspark.sql.functions as F
    
    f.write("5. Missing GUST Percentage (2024)\n")
    f.write("-----------------------------------\n")
    
    gust_2024 = weather_df.filter(col("YEAR") == 2024)
    
    missing_gust = gust_2024.groupBy("STATION") \
        .agg(
            (
                F.sum(col("GUST").isNull().cast("int")) /
                F.count("*") * 100
            ).alias("Missing_GUST_Percentage")
        )
    
    for row in missing_gust.collect():
        f.write(str(row) + "\n")
    
    f.write("\n")


    # -------------------------------------------------
    # 6. Monthly TEMP Stats (Cincinnati 2020)
    # -------------------------------------------------
    f.write("6. Monthly TEMP Stats (Cincinnati 2020)\n")
    f.write("----------------------------------------\n")

    cincy_2020 = weather_df.filter(
        (col("STATION") == 72429793812) &
        (col("YEAR") == 2020)
    )

    monthly_stats = cincy_2020.groupBy(month("DATE").alias("MONTH")) \
        .agg(
            avg("TEMP").alias("Mean"),
            expr("percentile_approx(TEMP, 0.5)").alias("Median"),
            stddev("TEMP").alias("StdDev")
        ).orderBy("MONTH")

    for row in monthly_stats.collect():
        f.write(str(row) + "\n")

    f.write("\n")


    # -------------------------------------------------
    # 7. Top 10 Lowest Wind Chill (Cincinnati 2017)
    # -------------------------------------------------
    f.write("7. Top 10 Lowest Wind Chill (Cincinnati 2017)\n")
    f.write("----------------------------------------------\n")

    cincy_2017 = weather_df.filter(
        (col("STATION") == 72429793812) &
        (col("YEAR") == 2017) &
        (col("TEMP") < 50) &
        (col("WDSP") > 3)
    )

    wc_df = cincy_2017.withColumn(
        "Wind_Chill",
        35.74 +
        0.6215 * col("TEMP") -
        35.75 * pow(col("WDSP"), 0.16) +
        0.4275 * col("TEMP") * pow(col("WDSP"), 0.16)
    )

    lowest_wc = wc_df.orderBy("Wind_Chill") \
        .select("DATE", "Wind_Chill") \
        .limit(10)

    for row in lowest_wc.collect():
        f.write(str(row) + "\n")

    f.write("\n")


    # -------------------------------------------------
    # 8. Extreme Weather Days (Florida)
    # -------------------------------------------------
    f.write("8. Extreme Weather Days (Florida)\n")
    f.write("-----------------------------------\n")

    florida = weather_df.filter(col("STATION") == 99495199999)
    extreme_days = florida.filter(col("FRSHTT") != 0)

    f.write(f"Extreme Weather Days Count: {extreme_days.count()}\n\n")


    # -------------------------------------------------
    # 9. Prediction for November & December 2024
    # -------------------------------------------------
    f.write("9. Prediction for November & December 2024\n")
    f.write("--------------------------------------------\n")

    f.write(f"Predicted November 2024 MAX: {nov_prediction}\n")
    f.write(f"Predicted December 2024 MAX: {dec_prediction}\n")


print("nguye3np_results.txt successfully created.")

nguye3np_results.txt successfully created.


In [38]:
Things to note for results analysis:

The Florida station (99495199999) reports zero precipitation across all years in the GSOD dataset. 
Therefore, the maximum yearly mean precipitation for Florida is 0.0 inches. 
This likely indicates that the station does not record precipitation data in this dataset.


For the prediction for November and december 2024:
    A simple linear regression model was implemented manually using covariance and variance formulas to 
    estimate the relationship between day-of-year and maximum temperature. 
    The model was trained using November and December data from 2022–2023. 
    Predictions were generated for mid-November and mid-December 2024.


SyntaxError: invalid syntax (3504232819.py, line 3)